In [11]:
%pip install python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
%pip install  "pinecone[grpc]" 


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
import os
from pathlib import Path
from dotenv import load_dotenv

base_dir = Path('/workspaces/Notebook-LLMs/pincone')
loaded = load_dotenv(base_dir / '.env')
if not loaded:
    loaded = load_dotenv(base_dir / '.env.example')

print('dotenv loaded:', loaded)
print('PINECONE_API_KEY found:', bool(os.getenv('PINECONE_API_KEY')))

dotenv loaded: True
PINECONE_API_KEY found: True


In [15]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import time

api_key=os.environ.get('PINECONE_API_KEY')
pc = Pinecone(api_key=api_key)

In [16]:
data = [
    {"id": "vec1", "text": "Apple is a popular fruit known for its sweetness and crisp texture."},
    {"id": "vec2", "text": "The tech company Apple is known for its innovative products like the iPhone."},
    {"id": "vec3", "text": "Many people enjoy eating apples as a healthy snack."},
    {"id": "vec4", "text": "Apple Inc. has revolutionized the tech industry with its sleek designs and user-friendly interfaces."},
    {"id": "vec5", "text": "An apple a day keeps the doctor away, as the saying goes."},
    {"id": "vec6", "text": "Apple Computer Company was founded on April 1, 1976, by Steve Jobs, Steve Wozniak, and Ronald Wayne as a partnership."}
]

# Convert the text into numerical vectors that Pinecone can index
embeddings = pc.inference.embed(
    model="multilingual-e5-large",
    inputs=[d['text'] for d in data],
    parameters={"input_type": "passage", "truncate": "END"}
)


In [17]:
print(embeddings)

EmbeddingsList(
  model='multilingual-e5-large',
  vector_type='dense',
  data=[
    {'vector_type': dense, 'values': [0.04931640625, -0.01328277587890625, ..., -0.0196380615234375, -0.010955810546875]},
    {'vector_type': dense, 'values': [0.032562255859375, -0.027862548828125, ..., -0.0200653076171875, -0.021026611328125]},
    ... (2 more embeddings) ...,
    {'vector_type': dense, 'values': [0.0312347412109375, -0.0185699462890625, ..., -0.02996826171875, -0.032989501953125]},
    {'vector_type': dense, 'values': [0.03955078125, -0.01013946533203125, ..., 0.0011348724365234375, -0.04296875]}
  ],
  usage={'total_tokens': 130}
)


In [18]:

index_name = "example-index"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(
            cloud='aws', 
            region='us-east-1'
        ) 
    ) 

while not pc.describe_index(index_name).status['ready']:
    time.sleep(1)

In [19]:
index = pc.Index("example-index")
records = []
for d, e in zip(data, embeddings):
    records.append({
        "id": d['id'],
        "values": e['values'],
        "metadata": {'text': d['text']}
    })

# Upsert the records into the index
index.upsert(
    vectors=records,
    namespace="example-namespace"
)

UpsertResponse(upserted_count=6, _response_info={'raw_headers': {'date': 'Mon, 13 Apr 2026 18:13:28 GMT', 'x-pinecone-request-lsn': '2', 'x-pinecone-request-logical-size': '25140', 'x-pinecone-request-latency-ms': '252', 'x-envoy-upstream-service-time': '253', 'x-pinecone-response-duration-ms': '258', 'server': 'envoy'}})

In [20]:
query = "Tell me about the tech company known as Apple."
query_embedding = pc.inference.embed(
    model="multilingual-e5-large",
    inputs=[query],
    parameters={
        "input_type": "query"
    }
)

# Search the index for the three most similar vectors
results = index.query(
    namespace="example-namespace",
    vector=query_embedding[0].values,
    top_k=3,
    include_values=False,
    include_metadata=True
)

print(results)

QueryResponse(matches=[{'id': 'vec2',
 'metadata': {'text': 'The tech company Apple is known for its innovative '
                      'products like the iPhone.'},
 'score': 0.8725984692573547,
 'sparse_values': None,
 'values': []}, {'id': 'vec4',
 'metadata': {'text': 'Apple Inc. has revolutionized the tech industry with '
                      'its sleek designs and user-friendly interfaces.'},
 'score': 0.8514811396598816,
 'sparse_values': None,
 'values': []}, {'id': 'vec6',
 'metadata': {'text': 'Apple Computer Company was founded on April 1, 1976, by '
                      'Steve Jobs, Steve Wozniak, and Ronald Wayne as a '
                      'partnership.'},
 'score': 0.85019451379776,
 'sparse_values': None,
 'values': []}], namespace='example-namespace', usage={'read_units': 1}, _response_info={'raw_headers': {'date': 'Mon, 13 Apr 2026 18:14:18 GMT', 'x-pinecone-max-indexed-lsn': '2', 'x-pinecone-request-latency-ms': '88', 'x-envoy-upstream-service-time': '90', 'x-pine